# MCD-rPPG ① 生成数据 → Google Drive（或任意本地目录）

**做什么**：把 HF 上 `kyegorov/mcd_rppg`（135 GB 原始 avi）**一段一段**地取下来，做官方同款预处理，产物直接落到**你的谷歌云盘**，随时断、随时续。

```
for 每一段记录（= 一个 avi + 一个 ppg_sync）:
    hf_hub_download 这一段 → /content 临时目录
    解码 → face_alignment 68 点 → 全局 bbox → 裁剪+凸包蒙版
    ├─ roi/{pid}_{step}_{view}.npz        32维/帧 特征 + 关键点 + ppg     ~1.2 MB/段
    ├─ preprocessed/mcd_rppg__{pid}__{step}.npz   video+landmarks+ppg     ~17 MB/段（可选）
    └─ faces/{pid}_{step}_{view}.mp4        128×128 人脸视频               ~10 MB/段（可选）
    把上面这些 copy 到 Drive → 删掉本地 avi → 下一段
```

**特性**
- 磁盘峰值 = 1 段视频（<100 MB），和总量无关
- **续跑**：以 Drive 上已存在的文件为准，重跑自动跳过
- **不重复提取**：以后训练只读 Drive（或你上传后的 HF），永远不再碰那 135 GB

**空间预算（免费 15 GB 要看清）**

| 你要的产物 | 每段 | 正脸 1200 段 | 三机位 3600 段 |
|---|---|---|---|
| `roi`（训主模型够用） | ~1.2 MB | **1.5 GB** | 4.3 GB |
| `official` npz @96px×600帧 | ~17 MB | **20 GB** ⚠️ | — |
| `faces` mp4 @128px | ~10 MB | **12 GB** ⚠️ | 36 GB |

→ 免费盘建议先只开 `SAVE_ROI=True`；要像素就用 `SAVE_OFFICIAL`（96px）分批做，做完一批立刻上传 HF 再从 Drive 删掉。

In [ ]:
%%bash
pip -q install "face-alignment==1.4.1" "huggingface_hub>=0.27" hf_transfer
python - <<'EOF'
import face_alignment, huggingface_hub, cv2, numpy, pandas, torch
print("face_alignment", face_alignment.__version__)
print("hub", huggingface_hub.__version__, "| torch", torch.__version__)
print("cv2", cv2.__version__, "| numpy", numpy.__version__, "| gpu", torch.cuda.is_available())
EOF

In [ ]:
# ===== 0.5 环境自检（10 秒，出问题先看这格）=====
import os, subprocess, torch

IN_COLAB = False
try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    pass

print("运行环境       :", "Colab" if IN_COLAB else "本地 / 容器 Jupyter（大概率没有 GPU 可切）")
print("torch          :", torch.__version__, "   ← 带 '+cpu' 后缀就是 CPU-only wheel")
print("torch.version.cuda:", torch.version.cuda, "  ← None = 装的是 CPU 版 torch")
print("cuda available :", torch.cuda.is_available(), "| device count:", torch.cuda.device_count())
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
try:
    out = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True, timeout=20).stdout.strip()
    print("nvidia-smi     :", out or "有命令但无输出（=没有卡）")
    HAS_NV = bool(out)
except FileNotFoundError:
    print("nvidia-smi     : 命令不存在（=这台机器没有 NVIDIA 驱动/显卡）")
    HAS_NV = False
except Exception as e:
    print("nvidia-smi     : 调用失败", e)
    HAS_NV = False

print()
if torch.cuda.is_available():
    print("✅ GPU 可用 →", torch.cuda.get_device_name(0), "→ 配置格里 LM_DEVICE 留 None 即可")
elif not IN_COLAB:
    print("❌ 结论：本地/容器环境无 GPU，切不了。两条路：")
    print("   ① 只做『冒烟验证』：执行下面的『0.6 CPU 冒烟预设』格，1 段 300 帧验证链路（几分钟）")
    print("   ② 全量必须换环境：真 Colab(T4) / Kaggle(每周 30h T4) / RunPod(约 $0.4/h)")
elif not HAS_NV:
    print("❌ 结论：Colab 运行时是 CPU 型 → 『运行时 → 更改运行时类型 → T4 GPU』")
    print("   → 『运行时 → 断开并删除运行时』→ 重新连接（必须删运行时，只重启内核无效）")
else:
    print("⚠️ 有显卡但 torch 用不了 → torch 被换成了 CPU wheel")
    print("   最省事：『运行时 → 断开并删除运行时』重连（Colab 出厂 torch 自带 cu，不用重装）")
    print("   或者：!pip install -q --force-reinstall torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121")

## 1. 配置（★ 常量在这里：路径 / 产物开关 / LIMIT 默认 5）

In [ ]:
import os
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")

# ================= 只改这一格 =================
DRIVE_ROOT = "/content/drive/MyDrive/mcd_rppg_ds"   # 产物落在你的谷歌云盘
VIEWS      = ["front"]        # 先 ["front"]；跑通后再 ["front","left","right"]
STEPS      = ["before", "after"]

SAVE_ROI      = True          # 32维/帧 特征 + 关键点 + ppg（训主模型用，~1.2MB/段）
SAVE_OFFICIAL = False         # 官方 drop-in npz：video+landmarks+ppg（~17MB/段）
SAVE_FACES    = False         # 裁剪人脸 mp4 128px（训视频模型用，~10MB/段）

FRAME_SIZE = 96               # SAVE_OFFICIAL 的方形边长
FACE_SIZE  = 128              # SAVE_FACES 的方形边长
CRF        = 12               # mp4 质量，0=无损
MAX_FRAMES = None             # None=全时长；SAVE_OFFICIAL 时必须 600（官方就是 600 帧）
COMPRESS   = True             # npz 是否压缩

BATCH_PER_ROUND = 10          # 一轮处理 N 段（只影响落盘节奏和断点粒度）
SESSION_SEC = 0               # >0 时跑够这么多秒就自动收工（防止会话超时白跑）
LIMIT      = 5                # ★ 先 5 段冒烟，看速度和体积，再改 0（全部）
BATCH      = 32               # face_alignment 批大小
LM_DEVICE  = None             # None = 自动探测（有 GPU 用 cuda，否则 cpu）
GPU_HINT_FRAMES = 600         # 没有 GPU 时自动把 MAX_FRAMES 收到 600，避免一段跑 20 分钟
WORK_DIR   = "/content/mcd_work"
SRC_REPO   = "kyegorov/mcd_rppg"
# ============================================

# ---- 设备自动探测：Colab 默认可能是 CPU runtime ----
import torch as _torch
HAS_CUDA = _torch.cuda.is_available()
if LM_DEVICE is None:
    LM_DEVICE = "cuda" if HAS_CUDA else "cpu"
print(f"torch {_torch.__version__} | cuda build {_torch.version.cuda} | 可用 GPU: {HAS_CUDA} → LM_DEVICE={LM_DEVICE}")
if LM_DEVICE == "cuda" and not HAS_CUDA:
    raise RuntimeError("LM_DEVICE='cuda' 但当前 torch 用不了 CUDA：菜单『运行时 → 更改运行时类型 → T4 GPU』，"
                       "或把 LM_DEVICE 设成 'cpu'，或重装匹配的 torch（见 notebook 最后『常见坑』第 9 条）")
if LM_DEVICE == "cpu":
    print("⚠️ 没有 GPU：face_alignment 在 Colab CPU 上约 0.2~0.5 s/帧 → 600 帧一段约 2~5 分钟，"
          "1200 段就是 40~100 小时，实际上不可行。建议先切到 T4 再跑（本节只是让你能冒烟验证代码）。")
    if MAX_FRAMES is None:
        MAX_FRAMES = GPU_HINT_FRAMES
        print(f"   已自动把 MAX_FRAMES 收到 {MAX_FRAMES}（只取前 {MAX_FRAMES} 帧），想全时长就手动改回 None")
    BATCH = min(BATCH, 8)          # CPU 上批太大只会占内存
    if LIMIT == 0:
        print("   建议先 LIMIT=5 量一下真实速度再放开")

if SAVE_OFFICIAL and MAX_FRAMES is None:
    MAX_FRAMES = 600          # 官方 rPPGDataset 要求同一 batch 内 T 一致，且它就是 600
for d in ["", "/src", "/out"]:
    os.makedirs(WORK_DIR + d, exist_ok=True)
print("drive :", DRIVE_ROOT)
print("views :", VIEWS, "| roi:", SAVE_ROI, "| official:", SAVE_OFFICIAL, "| faces:", SAVE_FACES)
print("limit :", LIMIT, "| frames:", MAX_FRAMES)

In [ ]:
# ===== 1.5 CPU 冒烟预设（可选）=====
# ⚠️ 这一格会把 LIMIT 压成 1！只在『验证链路』时跑；
#    要全量请跳过本格，去第『8.5 全量跑』格设 LIMIT=0（顺序很重要，8.5 在后）
# 前提：先跑过上面的配置格。约 1~3 分钟出结果，产物会真的写进 Drive。
LIMIT, MAX_FRAMES, BATCH = 1, 300, 4
SAVE_OFFICIAL, SAVE_FACES = False, False
print(f"冒烟模式：只跑 1 段、{MAX_FRAMES} 帧、batch {BATCH} → 预期 1~3 分钟")

## 2. 挂载谷歌云盘（这就是你说的"直接生成到谷歌云"）

In [ ]:
# Colab 就挂载云盘；本地 / 容器 Jupyter 就退回本地目录（这样同一份 notebook 两边都能跑）
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    print("✅ Colab：Drive 已挂载 →", DRIVE_ROOT)
except Exception as e:
    _old = DRIVE_ROOT
    DRIVE_ROOT = os.path.expanduser("~/mcd_rppg_ds")
    print(f"⚠️ 不是 Colab 或挂载失败（{type(e).__name__}），产物改存本地：{DRIVE_ROOT}")
    print(f"   （原来想写到 {_old}）")

for d in ["", "/roi", "/preprocessed", "/faces", "/index"]:
    os.makedirs(DRIVE_ROOT + d, exist_ok=True)
print("输出根目录 →", DRIVE_ROOT)
print(os.listdir(DRIVE_ROOT))

## 3. 工作清单（★ 改了 VIEWS/STEPS 必须重跑这一格）
- **`db.csv`（HF，3600 行）** = 工作清单，含 `video / ppg_sync / meta / ecg` 五个相对路径 + `camera/view` + 13 项生物指标
- **`mcd_rppg.csv`（GitHub，1200 行）** = 只用来拿 **fold**（按病人的 5 折）和做**对账金标准**（它只覆盖正脸）

⚠️ 不要拿 GitHub 那个 csv 当清单——它里面没有 left/right 那 2400 段，也没有 HF 上的 avi 路径。

In [ ]:
import io, json, math, re, shutil, subprocess, sys, time
import datetime as dt
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import scipy.signal
import torch
from huggingface_hub import HfApi, hf_hub_download, login

SRC = SRC_REPO
HF_TOKEN = os.environ.get("HF_TOKEN")
try:
    from google.colab import userdata
    HF_TOKEN = HF_TOKEN or userdata.get("HF_TOKEN")
except Exception:
    pass
if HF_TOKEN:
    login(token=HF_TOKEN)
    print("HF 已登录（只用于避免匿名限流）")
api = HfApi()

db = pd.read_csv(hf_hub_download(SRC, "db.csv", repo_type="dataset"))
db["key"] = [f"{int(p):04d}_{s}_{v}" for p, s, v in zip(db.patient_id, db.step, db.view)]
db = db[db["view"].isin(VIEWS) & db["step"].isin(STEPS)].reset_index(drop=True)

fold_map = {}
try:
    off = pd.read_csv("https://raw.githubusercontent.com/ksyegorov/mcd_rppg/main/mcd_rppg.csv")
    off["pid"] = off["file"].str.extract(r"mcd_rppg__(\d+)__")[0].astype(int)
    off["stp"] = off["file"].str.extract(r"mcd_rppg__\d+__(\w+)\.npz")[0]
    REF = {(int(a), b): r for (a, b), r in
           zip(zip(off.pid, off.stp), off.itertuples())}
    fold_map = {k: int(v.fold) for k, v in REF.items()}
    print("官方 fold（按病人）:", len(fold_map), "条")
except Exception as e:
    REF = {}
    print("官方 fold 取不到，改为 patient_id % 5 :", e)

db["fold"] = [fold_map.get((int(p), s), int(p) % 5) for p, s in zip(db.patient_id, db.step)]

BIOMARKERS = ["upper_ap", "lower_ap", "saturation", "temperature", "hemoglobin",
              "glycated_hemoglobin", "cholesterol", "respiratory", "rigidity",
              "pulse", "stress", "age", "bmi", "sex", "weight", "height"]


def build_budget():
    mb = {"front": 78.0, "left": 29.0, "right": 16.0}
    n, T = len(db), (4800 if MAX_FRAMES is None else int(MAX_FRAMES))
    dl = sum(mb.get(v, 40.0) for v in db.view) / 1000.0
    k = T / 4800.0
    roi = n * 1.2 * k / 1000
    off_T = min(T, 600)          # 官方 drop-in npz 固定 600 帧（rPPGDataset 要求 T 一致）
    off_gb = n * (off_T * FRAME_SIZE * FRAME_SIZE * 3 / 1e6 * (0.85 if COMPRESS else 1.0)) / 1000 if SAVE_OFFICIAL else 0
    fac = n * 10.0 * k * (FACE_SIZE / 128.0) ** 2 / 1000 if SAVE_FACES else 0
    print(f"记录数           : {n}   (平均 {n and sum(mb.get(v,40.0) for v in db.view)/n:.0f} MB/段，需下载一次)")
    print(f"累计下载流量     : {dl:7.1f} GB   （磁盘峰值 {max(mb.get(v,40.0) for v in db.view):.0f} MB × 每轮 1 段）")
    print(f"Drive — roi      : {roi:7.2f} GB")
    if SAVE_OFFICIAL: print(f"Drive — official : {off_gb:7.2f} GB")
    if SAVE_FACES:    print(f"Drive — faces    : {fac:7.2f} GB")
    tot = roi + off_gb + fac
    print(f"Drive — 合计     : {tot:7.2f} GB" + ("   ⚠️ 超 12 GB，免费盘吃紧" if tot > 12 else "   ✅ 免费 15 GB 内"))


build_budget()

## 4. 工具函数
关键点用 `face_alignment`(FAN, 68 点)，与作者同一套 → `rppglib/models/SCNN_8rois.py` 里的 8 个 ROI 索引可以原样照搬。
`POS` 按 `torchPOS.py` 语义实现（1.6 s 滑窗 + overlap-add + 0.75–3 Hz 带通），保证特征与官方一致。

In [ ]:
import face_alignment

_FA = None


def to_tensor(frames):
    # face_alignment 批量接口要 (B,3,H,W) 的 RGB tensor，不是 list[np.ndarray]
    return torch.stack([torch.from_numpy(np.ascontiguousarray(f)).permute(2, 0, 1) for f in frames])


def get_fa():
    global _FA
    if _FA is None:
        LT = getattr(face_alignment.LandmarksType, "TWO_D", None) or face_alignment.LandmarksType._2D
        dev = LM_DEVICE
        try:
            _FA = face_alignment.FaceAlignment(LT, device=dev, flip_input=False)
        except Exception as e:
            if str(dev) == "cpu":
                raise
            print(f"⚠️ face_alignment({dev}) 初始化失败：{type(e).__name__}: {e}")
            print("   自动退回 CPU（能跑但慢 10~30 倍；想用 GPU 见『常见坑』第 9 条）")
            globals()["LM_DEVICE"] = "cpu"
            _FA = face_alignment.FaceAlignment(LT, device="cpu", flip_input=False)
        print("face_alignment ready on", globals().get("LM_DEVICE", dev))
    return _FA


# === 与 rppglib/models/SCNN_8rois.py 完全一致（68 点索引）===
FACE              = [1,2,3,4,5,6,8,9,10,11,12,13,14,15,16,17,27,26,25,24,23,22,21,20,19,18]
NOSE              = [22,23,43,36,35,34,33,32,40]
LEFT_UPPER_CHEEK  = [1,37,42,41,40,32,3,2]
RIGHT_UPPER_CHEEK = [17,46,47,48,43,36,15,16]
LEFT_LOWER_CHEEK  = [3,32,51,50,49,5,4]
RIGHT_LOWER_CHEEK = [15,36,53,54,55,13,14]
LIPS              = [49,50,51,52,53,54,55,56,57,58,59,60]
CHIN              = [5,6,7,8,9,10,11,12,13,56,57,58,59,60]
REGIONS = [FACE, NOSE, LEFT_UPPER_CHEEK, RIGHT_UPPER_CHEEK,
           LEFT_LOWER_CHEEK, RIGHT_LOWER_CHEEK, LIPS, CHIN]


def iter_frame_batches(path, start, end, bs):
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        raise RuntimeError("cannot open " + str(path))
    if start:
        cap.set(cv2.CAP_PROP_POS_FRAMES, start)
    i, buf = start, []
    while end is None or i < end:
        ok, fr = cap.read()
        if not ok:
            break
        buf.append(fr[..., ::-1].copy())      # BGR -> RGB
        i += 1
        if len(buf) == bs:
            yield buf
            buf = []
    if buf:
        yield buf
    cap.release()


def region_rgbs(img, lms):
    H, W = img.shape[:2]
    out = np.zeros((8, 3), np.float32)
    for k, seg in enumerate(REGIONS):
        mask = np.zeros((H, W), np.uint8)
        cv2.fillPoly(mask, [np.round(np.asarray(lms)[seg]).astype(np.int32)], 255)
        px = img[mask > 125]
        if px.size:
            out[k] = px.mean(0)
    return out


def mask_convex(frame, pts):
    m = np.zeros(frame.shape[:2], np.uint8)
    cv2.fillPoly(m, [cv2.convexHull(np.round(pts).astype(np.int32))], 255)
    frame *= (m > 125)[..., None]
    return frame


def pos_signal(rgb, fs):
    Cn = rgb / (rgb.mean(0, keepdims=True) + 1e-6)
    S = Cn @ np.array([[0, 1, -1], [-2, 1, 1], [-1, 2, -1]], np.float64).T
    n = len(Cn)
    L = min(max(int(round(1.6 * fs)), 8), n)
    win = np.lib.stride_tricks.sliding_window_view(S, L, axis=0)
    s0, s1 = win[:, 0, :], win[:, 1, :]
    alpha = s0.std(1) / (s1.std(1) + 1e-4)
    h = s0 + alpha[:, None] * s1
    h = h - h.mean(1, keepdims=True)
    H = np.zeros(n)
    m = n - L + 1
    for k in range(L):
        H[k:k + m] += h[:, k]
    if n > 16:
        sos = scipy.signal.butter(1, [0.75 / (fs / 2), 3.0 / (fs / 2)], btype="bandpass", output="sos")
        H = scipy.signal.sosfiltfilt(sos, H)
    return H.astype(np.float32)


def bandpass(x, fs, lo=0.5, hi=8.0, order=4):
    sos = scipy.signal.butter(order, [lo / (fs / 2), min(hi, fs / 2 * 0.99) / (fs / 2)],
                              btype="bandpass", output="sos")
    return scipy.signal.sosfiltfilt(sos, np.asarray(x, np.float64)).astype(np.float32)


def hr_bpm(x, fs, lo=0.5, hi=3.5):
    x = np.asarray(x, np.float64)
    x = x - x.mean()
    f, p = scipy.signal.periodogram(x, fs=fs, nfft=2048, detrend=False)
    m = (f >= lo) & (f <= hi)
    if not m.any():
        return 0.0, 0.0
    return float(f[m][p[m].argmax()] * 60), float(10 * np.log10(p[m].max() / (p[m].mean() + 1e-12)))


def parse_meta(path, T, fps):
    try:
        ts = []
        with open(path) as f:
            for line in f:
                parts = line.split()
                if len(parts) >= 2:
                    ts.append(dt.datetime.fromisoformat(parts[1]).timestamp())
        ts = np.asarray(ts, np.float64)
        if len(ts) < T:
            ts = np.pad(ts, (0, T - len(ts)), mode="edge")
        ts = ts[:T]
        return (ts - ts[0]).astype(np.float64), True
    except Exception:
        return np.arange(T, dtype=np.float64) / fps, False


def encode_cmd(out_path, size, fps):
    return ["ffmpeg", "-y", "-loglevel", "error",
            "-f", "rawvideo", "-pix_fmt", "rgb24", "-s", f"{size}x{size}",
            "-r", f"{fps:.6f}", "-i", "-",
            "-c:v", "libx264", "-preset", "veryfast", "-crf", str(CRF),
            "-pix_fmt", "yuv444p", out_path]


def du(path):
    tot = 0
    for root, _, fs in os.walk(path):
        for f in fs:
            try:
                tot += os.path.getsize(os.path.join(root, f))
            except Exception:
                pass
    return tot / 1e9


def _probe_fps(path):
    """ffprobe 读 avg_frame_rate（比 cv2 可靠，但 AVI 头也可能写坏）"""
    try:
        out = subprocess.run(["ffprobe", "-v", "error", "-select_streams", "v:0",
                              "-show_entries", "stream=avg_frame_rate",
                              "-of", "csv=p=0", str(path)],
                             capture_output=True, text=True, timeout=60).stdout.strip()
        num, den = (out.split("/") + ["1"])[:2]
        v = float(num) / float(den) if float(den) else 0.0
        return v
    except Exception:
        return 0.0


def resolve_fps(path, T, ts, ts_ok):
    """三方交叉验证帧率。meta 里的采集端时间戳是唯一"真值"，优先用它。
    起因：部分 FullHDwebcam 的 avi 头把 29.97 写成 24，cv2 照抄，
    而 ppg_sync 是逐帧对齐的 → fs 错 → 心率被低估 20%。"""
    cands = {}
    cap = cv2.VideoCapture(str(path))
    c = cap.get(cv2.CAP_PROP_FPS)
    cap.release()
    cands["cv2"] = float(c) if c and c > 1 else 0.0
    cands["ffprobe"] = _probe_fps(path)
    if ts_ok and T > 1 and ts[-1] > ts[0]:
        cands["meta"] = (T - 1) / float(ts[-1] - ts[0])     # 帧数 / 真实时长
    else:
        cands["meta"] = 0.0
    for k in ("meta", "ffprobe", "cv2"):                    # 优先级
        v = cands[k]
        if 15.0 <= v <= 61.0:
            return float(v), k, cands
    return 30.0, "default", cands


## 5. 单段处理 + Drive 落地

- `expected_paths()` 决定"这段算不算做完"→ 这就是续跑依据
- `put()` 先 copy 到 Drive、校验大小、再删本地 → Drive 写失败不会丢中间结果
- 两遍解码：第一遍只取关键点/ROI（定出全程 bbox），第二遍才裁像素。**两遍都是读本地文件，不产生额外流量**

In [ ]:
def process_record(r):
    t0 = time.time()
    key = r.key
    info = dict(key=key, patient_id=int(r.patient_id), step=r.step, view=r.view,
                camera=r.camera, fold=int(r.fold), status="ok", err="")
    for b in BIOMARKERS:
        info[b] = getattr(r, b, None)
    src = f"{WORK_DIR}/src"

    # ppg_sync：★ 用第 0 列（ppg_sync_npy 那份极性是反的：col0 + npy ≡ 236）
    p = hf_hub_download(SRC, r.ppg_sync, repo_type="dataset", local_dir=src)
    ppg_all = np.loadtxt(p, ndmin=2)[:, 0].astype(np.float32)
    os.unlink(p)
    n_max = len(ppg_all) if MAX_FRAMES is None else min(int(MAX_FRAMES), len(ppg_all))

    # meta：每帧采集端时间戳 —— 用来推算"真实帧率"（avi 头可能是错的）
    m = hf_hub_download(SRC, r.meta, repo_type="dataset", local_dir=src)

    v = hf_hub_download(SRC, r.video, repo_type="dataset", local_dir=src)
    cap = cv2.VideoCapture(v)
    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps_hdr = cap.get(cv2.CAP_PROP_FPS)
    cap.release()

    # ---------- pass 1：68 点 + 8 ROI 均值（ROI 在整帧上算，裁剪只是平移）----------
    fa = get_fa()
    lms_l, roi_l, prev, miss = [], [], np.zeros((68, 2), np.float32), 0
    for batch in iter_frame_batches(v, 0, n_max, BATCH):
        preds = fa.get_landmarks_from_batch(to_tensor(batch))
        for img, lm in zip(batch, preds):
            if lm is None or len(lm) == 0:      # 批量接口找不到脸时返回 []
                miss += 1
                lm = prev
            else:
                lm = np.asarray(lm, np.float32)[:68]
                prev = lm
            lms_l.append(lm.copy())
            if SAVE_ROI:
                roi_l.append(region_rgbs(img, lm))
    T = len(lms_l)
    if T < 30:
        raise RuntimeError(f"only {T} frames decoded")
    lms = np.asarray(lms_l, np.float32)
    ppg = ppg_all[:T]
    roi = np.asarray(roi_l, np.float32) if SAVE_ROI else None

    # ---------- 帧率：meta 时间戳反推（真值）> ffprobe > cv2 ----------
    ts, ts_ok = parse_meta(m, T, fps_hdr if fps_hdr and fps_hdr > 1 else 30.0)
    os.unlink(m)
    fps, fps_src, fps_cands = resolve_fps(v, T, ts, ts_ok)
    info.update(fps_hdr=round(float(fps_hdr or 0), 2), fps_probe=round(fps_cands["ffprobe"], 2),
                fps_meta=round(fps_cands["meta"], 2), fps_src=fps_src)
    if fps_hdr and fps_hdr > 1 and abs(fps_hdr - fps) > 2.0:
        info["err"] = (f"fps头{fps_hdr:.2f}≠实际{fps:.2f}(用{fps_src})"
                       + (";" + info["err"] if info["err"] else ""))

    lm_i = lms.round().astype(np.int16)
    x0, x1 = max(0, int(lm_i[..., 0].min())), min(W, int(lm_i[..., 0].max()))
    y0, y1 = max(0, int(lm_i[..., 1].min())), min(H, int(lm_i[..., 1].max()))
    if x1 - x0 < 16 or y1 - y0 < 16:
        raise RuntimeError("bad bbox")
    hr_t, snr_t = hr_bpm(bandpass(ppg, fps), fps)
    info.update(T=T, face_square=int((y1 - y0) * (x1 - x0)), fps=round(float(fps), 4),
                miss=int(miss), hr_ppg=round(hr_t, 2), snr=round(snr_t, 2),
                hr_tone=getattr(r, "pulse", None))
    if getattr(r, "pulse", None):
        info["hr_gap"] = round(abs(hr_t - float(r.pulse)), 1)

    done = []

    # ---------- roi npz ----------
    if SAVE_ROI:
        f = f"{WORK_DIR}/out/roi_{key}.npz"
        (np.savez_compressed if COMPRESS else np.savez)(
            f,
            landmarks=lm_i, ppg=ppg, roi_rgb=roi,
            roi_pos=np.stack([pos_signal(roi[:, k, :], fps) for k in range(8)], 1),
            fps=np.float32(fps), ts=ts, face_square=np.int32(info["face_square"]),
            crop=np.array([x0, y0, x1, y1], np.int32), n_missing=np.int32(miss),
            patient_id=np.int32(r.patient_id), step=np.array(r.step), view=np.array(r.view),
            camera=np.array(r.camera), hr_ppg=np.float32(hr_t), ts_ok=np.bool_(ts_ok),
            fps_src=np.array(fps_src),
        )
        done.append(put(f, f"roi/{key}.npz"))

    # ---------- pass 2：裁剪 + 凸包蒙版（+ official npz / mp4）----------
    if SAVE_OFFICIAL or SAVE_FACES:
        cw, ch = x1 - x0, y1 - y0
        lm_off = np.array([x0, y0], np.float32)
        proc, mp4 = None, None
        if SAVE_FACES:
            mp4 = f"{WORK_DIR}/out/faces_{key}.mp4"
            proc = subprocess.Popen(encode_cmd(mp4, FACE_SIZE, fps), stdout=subprocess.DEVNULL,
                                    stderr=subprocess.DEVNULL, stdin=subprocess.PIPE)
        keep = min(T, int(MAX_FRAMES)) if (SAVE_OFFICIAL and MAX_FRAMES) else T
        frames = [] if SAVE_OFFICIAL else None
        n = 0
        for batch in iter_frame_batches(v, 0, T, 32):
            for img in batch:
                c = img[y0:y1, x0:x1].copy()
                c = mask_convex(c, lms[n] - lm_off)
                if SAVE_OFFICIAL and n < keep:
                    frames.append(cv2.resize(c, (FRAME_SIZE, FRAME_SIZE), interpolation=cv2.INTER_AREA))
                if proc is not None:
                    cf = cv2.resize(c, (FACE_SIZE, FACE_SIZE), interpolation=cv2.INTER_AREA)
                    proc.stdin.write(np.ascontiguousarray(cf).tobytes())
                n += 1
        if proc is not None:
            proc.stdin.close()
            proc.wait()
            done.append(put(mp4, f"faces/{key}.mp4"))
        if n != T:
            info["err"] = (f"pass2 {n}!={T};" + info["err"]) if info["err"] else f"pass2 {n}!={T}"

        if SAVE_OFFICIAL:
            nk = len(frames)
            lm_crop = (lms[:nk] - lm_off) * np.array([FRAME_SIZE / cw, FRAME_SIZE / ch], np.float32)
            f = f"{WORK_DIR}/out/off_{key}.npz"
            suf = "" if r.view == "front" else f"_{r.view}"
            (np.savez_compressed if COMPRESS else np.savez)(
                f, video=np.stack(frames), landmarks=lm_crop.round().astype(np.int16),
                ppg=ppg[:nk], fps=np.float32(fps))
            done.append(put(f, f"preprocessed/mcd_rppg__{int(r.patient_id)}__{r.step}{suf}.npz"))

    if os.path.exists(v):
        os.unlink(v)
    info["sec"] = round(time.time() - t0, 1)
    info["files"] = len(done)
    return done, info


## 6. 主循环（★ 跑全量就是跑这一格，可反复重跑续跑）
每一轮：把 N 段处理完 → 落到 Drive → 打印速度与预计剩余。中途断了直接重跑这一格，已完成的会自动跳过。

**轮数 = ceil(待处理段数 / `BATCH_PER_ROUND`)**。所以 `LIMIT=5` + `BATCH_PER_ROUND=10` 只会有 **1 轮**（这是正常的）；
要全量就 `LIMIT=0`。如果 5 条全失败，说明是「每一条都过不去」（比如设备/路径问题），不是循环没跑。

In [ ]:
def chunked(seq, n):
    for i in range(0, len(seq), n):
        yield i // n, seq[i:i + n]


todo_all = db[~db.apply(is_done, axis=1)].reset_index(drop=True)
n_done = len(db) - len(todo_all)
todo = todo_all.head(LIMIT) if LIMIT else todo_all     # LIMIT=0 → todo = todo_all（全量）
print(f"总记录 {len(db)} | 已完成 {n_done} | 本次待处理 {len(todo)}")

t_start, n_ok, all_rows = time.time(), 0, []
RUN_TAG = time.strftime("%m-%d %H:%M")          # 本次会话标记，用于溯源
_seen = set()                                   # 本次会话真实尝试过的 key

for bi, chunk in chunked(todo, BATCH_PER_ROUND):     # 每轮 BATCH_PER_ROUND 段
    round_rows = []
    for r in chunk.itertuples():
        try:
            files, info = process_record(r)          # 单段：下载 → 提取 → 落 Drive → 删本地 avi
            info["ts"], info["origin"] = RUN_TAG, "this-run"
            _seen.add(r.key)
            round_rows.append(info)
            n_ok += 1
            print(f'  ✓ {info["key"]}  T={info["T"]} fps={info["fps"]}'
                  f'({info.get("fps_src", "?")}) HR={info["hr_ppg"]}/{info["hr_tone"]}'
                  f'±{info.get("hr_gap", "?")} miss={info["miss"]} '
                  f'{info["sec"]}s{"  !" + info["err"] if info["err"] else ""}')
        except Exception as e:
            round_rows.append(dict(key=r.key, patient_id=int(r.patient_id), step=r.step,
                                   view=r.view, status="error", err=str(e)[:200],
                                   ts=RUN_TAG, origin="this-run"))
            _seen.add(r.key)
            print("  ✗", r.key, type(e).__name__, str(e)[:140])
            if n_ok == 0 and len(round_rows) == 1:
                import traceback as _tb
                _tb.print_exc()                      # 只对第一条失败打印完整栈

    # 每轮落一次盘（只写本轮），并清空本轮下载的源文件
    pd.DataFrame(round_rows).to_parquet(f"{WORK_DIR}/out/log_{bi:05d}.parquet")
    all_rows += round_rows
    freed = 0
    for f in Path(WORK_DIR, "src").rglob("*"):       # 递归：hf_hub_download 会建 video/ ppg_sync/ 等子目录
        if f.is_file():
            try:
                freed += f.stat().st_size
                f.unlink()
            except Exception:
                pass

    el = time.time() - t_start
    rate = el / max(n_ok, 1)
    print(f"[round {bi}] 本轮 {len(round_rows)} 段 | 累计 {n_ok} | {el/60:.1f} min | "
          f"均 {rate:.0f} s/段 | 释放 {freed/1e6:.0f} MB | 临时盘 {du(WORK_DIR)*1000:.0f} MB | "
          f"剩余 {(len(todo)-n_ok)*rate/3600:.1f} h")
    if SESSION_SEC and (time.time() - t_start) > SESSION_SEC:
        print(f"[会话预算到] 已用 {(time.time()-t_start)/60:.0f} 分钟，收工。重跑这一格继续，已完成的会自动跳过")
        break

rows = all_rows
print(f"\n本次会话完成 {n_ok} 段（失败 {sum(1 for x in all_rows if x.get('status') == 'error')} 段）")


## 7. 对账：和官方 `mcd_rppg.csv` 比（强烈建议先跑）
官方 csv 里每行的 `video_shape / face_square / ppg_*` 其实是**前 600 帧**的统计。能对上 → 说明你的预处理与作者一致。

In [ ]:
def verify_repo_compat(pid=1020, step="after"):
    if not REF:
        print("没有官方 csv，跳过")
        return
    row = REF[(pid, step)]
    r = db[(db.patient_id == pid) & (db.step == step) & (db.view == "front")].iloc[0]
    src = f"{WORK_DIR}/src"
    v = hf_hub_download(SRC, r.video, repo_type="dataset", local_dir=src)
    p = hf_hub_download(SRC, r.ppg_sync, repo_type="dataset", local_dir=src)
    ppg = np.loadtxt(p, ndmin=2)[:600, 0].astype(np.float32)

    fa = get_fa()
    lms_l = []
    for batch in iter_frame_batches(v, 0, 600, BATCH):
        for img, pr in zip(batch, fa.get_landmarks_from_batch(to_tensor(batch))):
            lms_l.append(np.zeros((68, 2), np.float32) if (pr is None or len(pr) == 0)
                         else np.asarray(pr, np.float32)[:68])
    lms = np.asarray(lms_l).round().astype(np.int16)
    x0, x1 = int(lms[..., 0].min()), int(lms[..., 0].max())
    y0, y1 = int(lms[..., 1].min()), int(lms[..., 1].max())
    print(f"ours : shape=({len(lms)}, {y1-y0}, {x1-x0}, 3)  face_square={(y1-y0)*(x1-x0)}  "
          f"ppg {ppg.min():.3f}/{ppg.max():.3f}/{ppg.mean():.3f}/{ppg.std():.3f}")
    print(f"repo : shape={row.video_shape}  face_square={row.face_square}  "
          f"ppg {row.ppg_min}/{row.ppg_max}/{row.ppg_mean:.3f}/{row.ppg_std:.3f}")
    for f in (v, p):
        os.unlink(f)


try:
    verify_repo_compat(1020, "after")
except Exception as e:
    print("对账跳过:", type(e).__name__, e)

## 7.5 肉眼验收：把裁剪 + 蒙版画出来（1 段，几秒）

数字对得上不代表**裁得对**。这格取 1 段视频的前 90 帧，把「原图 + 68 点头罩 + 裁剪蒙版」拼成一张图，
顺便标出 8 个 ROI 的采样点。跑一次确认：脸在框里、蒙版没切掉下巴/额头、关键点没漂。

In [ ]:
def preview(key=None, n=12, n_frames=90):
    import matplotlib.pyplot as plt
    key = key or db.key.iloc[0]
    r = db[db.key == key].iloc[0]
    v = hf_hub_download(SRC, r.video, repo_type="dataset", local_dir=f"{WORK_DIR}/src")

    fa, imgs, lms = get_fa(), [], []
    for batch in iter_frame_batches(v, 0, n_frames, 16):
        for img, lm in zip(batch, fa.get_landmarks_from_batch(to_tensor(batch))):
            imgs.append(img)
            lms.append(np.asarray(lm, np.float32)[:68] if (lm is not None and len(lm))
                       else np.zeros((68, 2), np.float32))
    lms = np.asarray(lms, np.float32)
    li = lms.round().astype(np.int16)
    x0, x1 = max(0, int(li[..., 0].min())), int(li[..., 0].max())
    y0, y1 = max(0, int(li[..., 1].min())), int(li[..., 1].max())
    print(f"{key}: frames={len(imgs)} bbox=({x0},{y0})-({x1},{y1}) "
          f"crop={y1-y0}x{x1-x0} face_square={(y1-y0)*(x1-x0)}")
    print("missing landmarks:", int((np.abs(lms).sum((1, 2)) == 0).sum()), "/", len(lms))

    sel = np.linspace(0, len(imgs) - 1, n).astype(int)
    cols = 4
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(3.2 * cols, 2.6 * rows))
    axes = np.atleast_2d(axes)
    for ax, idx in zip(axes.ravel(), sel):
        crop = mask_convex(imgs[idx][y0:y1, x0:x1].copy(), lms[idx] - np.array([x0, y0], np.float32))
        vis = crop.astype(np.float32) / 255.0
        ax.imshow(vis)
        for seg in REGIONS:                      # 8 个 ROI 的顶点
            pts = lms[idx][seg] - np.array([x0, y0], np.float32)
            ax.scatter(pts[:, 0], pts[:, 1], s=3, c="lime", alpha=0.8)
        ax.set_title(f"frame {idx}", fontsize=8)
        ax.axis("off")
    for ax in axes.ravel()[len(sel):]:
        ax.axis("off")
    plt.tight_layout()
    out = f"{WORK_DIR}/preview_{key}.png"
    plt.savefig(out, dpi=110)
    plt.show()
    os.unlink(v)
    print("保存:", out)


preview()

## 7.6 数据集自检 `verify_dataset()`（不重新下载视频，只扫 Drive 上的 npz）

分三层查，一层不过就别往下跑：

| 层 | 查什么 | 常见病因 |
|---|---|---|
| **结构** | 6 个 key 齐不齐、各数组长度是否都等于 T、dtype | 写 npz 时被杀 / 中途断电 |
| **数值** | ppg 标准差≈0（信号平了）、roi_rgb 全黑或全白、NaN、landmarks 跑出画面外 640×480 | 蒙版错、关键点漂到画面外、坏帧 |
| **对账** | 和官方 `mcd_rppg.csv` 的 `face_square` / `ppg_min/max/mean/std` 比（仅正脸、且 T≥600 时） | 预处理与作者不一致 |

最后会把 QC 表存成 `qc_report.csv` 方便前后对比。

In [ ]:
def verify_dataset(sample=None, against_off=True, verbose=True):
    import glob as _glob
    files = sorted(_glob.glob(f"{DRIVE_ROOT}/roi/*.npz"))
    if sample:
        files = files[:int(sample)]
    if not files:
        print("Drive 上没有 roi/*.npz")
        return None
    rows, bad = [], []
    for i, f in enumerate(files):
        key = os.path.basename(f)[:-4]
        try:
            z = np.load(f)
            need = ["landmarks", "ppg", "roi_rgb", "roi_pos", "fps", "ts"]
            miss = [k for k in need if k not in z.files]
            T = int(len(z["ppg"])) if "ppg" in z.files else 0
            r = dict(key=key, T=T, fps=float(z["fps"]) if "fps" in z.files else np.nan,
                     n_missing_keys=len(miss), missing=",".join(miss),
                     len_ok=False, nan=False, ppg_std=np.nan, rgb_mean=np.nan,
                     lm_out=0, face_square=int(z["face_square"]) if "face_square" in z.files else -1,
                     ppg_min=np.nan, ppg_max=np.nan, ppg_mean=np.nan)
            if not miss:
                r["len_ok"] = all(len(z[k]) == T for k in need if k != "fps")
                r["nan"] = bool(np.isnan(np.asarray(z["roi_rgb"], np.float64)).any()
                                or np.isnan(np.asarray(z["ppg"], np.float64)).any())
                r["ppg_std"] = float(np.std(z["ppg"]))
                r["rgb_mean"] = float(np.mean(z["roi_rgb"]))
                lm = z["landmarks"]
                r["lm_out"] = int(((lm[..., 0] < 0) | (lm[..., 0] >= 640) |
                                   (lm[..., 1] < 0) | (lm[..., 1] >= 480)).sum())
                ppg = np.asarray(z["ppg"], np.float32)
                r["ppg_min"], r["ppg_max"] = float(ppg.min()), float(ppg.max())
                r["ppg_mean"] = float(ppg.mean())
            rows.append(r)
            why = []
            if r["n_missing_keys"]: why.append("缺key:" + r["missing"])
            if not r["len_ok"]: why.append("长度不一致")
            if r["nan"]: why.append("含NaN")
            if r["ppg_std"] < 1e-3: why.append("ppg平坦")
            if not (1 < r["rgb_mean"] < 254): why.append("画面全黑/全白")
            if r["lm_out"] > 0.02 * max(T, 1) * 68: why.append("关键点出界")
            if not (20 <= r["fps"] <= 61): why.append("fps异常")
            if why:
                bad.append((key, ";".join(why)))
        except Exception as e:
            bad.append((key, f"{type(e).__name__}: {e}"))
        if verbose and (i + 1) % 200 == 0:
            print(f"  已扫 {i+1}/{len(files)}")

    df = pd.DataFrame(rows)
    print(f"\n扫描 {len(files)} 个 npz | 有问题 {len(bad)} 个")
    if len(df):
        print(df[["T", "fps", "ppg_std", "rgb_mean", "lm_out", "face_square"]].describe().round(2).to_string())
    if bad:
        print("\n前 20 个问题文件：")
        for k, w in bad[:20]:
            print(f"  {k}: {w}")

    if against_off and REF and len(df):
        cmp = []
        for r in df.itertuples():
            pid, rest = r.key.split("_", 1)
            step, view = rest.rsplit("_", 1)
            ref = REF.get((int(pid), step))
            if ref is None or view != "front" or r.T < 600:
                continue
            cmp.append(dict(key=r.key, face_square_delta=r.face_square - int(ref.face_square),
                            ppg_mean_delta=round(r.ppg_mean - ref.ppg_mean, 3)))
        if cmp:
            c = pd.DataFrame(cmp)
            print(f"\n对账（{len(c)} 条正脸，和官方 csv 比）：")
            print(c[["face_square_delta", "ppg_mean_delta"]].describe().round(3).to_string())
            exact = int((c.face_square_delta == 0).sum())
            print(f"  face_square 完全一致: {exact}/{len(c)}"
                  f"  ({'✅ 预处理与作者一致' if exact == len(c) else '⚠️ 有偏差，看 delta 分布'})")
    try:
        out = os.path.join(DRIVE_ROOT, "qc_report.csv")
        df.to_csv(out, index=False)
        print("\nQC 表已存:", out)
    except Exception as e:
        print("QC 表存盘失败:", e)
    return df


qc = verify_dataset()

## 7.7 作废旧产物（★ 修了 fps 判定后，之前跑的都要重做）

之前 `fps` 是照抄 avi 头的，部分正脸视频头里写的是 24（真实 29.97），导致：
`ppg` 的带通频率、`POS` 的窗长、`hr_ppg` 全都不对，`roi_pos` 也偏。

**建议**：确认下面这格后执行一次，把 Drive 上的旧产物清掉重建。

```python
PURGE = True    # 改成 True 才会真的删
```
只删 `roi/`、`preprocessed/`、`faces/` 和 `index.parquet`，**不碰你别的文件**。

In [ ]:
# ===== 7.7 清空重建（fps 修好后建议执行一次）=====
PURGE = False          # ← 确认无误后改成 True 再跑

if PURGE:
    import shutil
    for sub in ["roi", "preprocessed", "faces"]:
        d = os.path.join(DRIVE_ROOT, sub)
        n = len(os.listdir(d)) if os.path.isdir(d) else 0
        if os.path.isdir(d):
            shutil.rmtree(d)
        os.makedirs(d, exist_ok=True)
        print(f"清空 {sub:13s} 删掉 {n} 个文件")
    ip = os.path.join(DRIVE_ROOT, "index.parquet")
    if os.path.exists(ip):
        os.unlink(ip)
        print("删掉 index.parquet")
    for f in Path(WORK_DIR, "out").glob("log_*.parquet"):
        f.unlink()
    print("\n✅ 已清空。接下来：跑『3. 工作清单』→『8.5 全量跑』→『8.6 确认』→『6. 主循环』")
else:
    print("PURGE = False，什么都没删。确认要重建就把上面改成 True 再跑这一格。")

## 8. 汇总索引 + 完成度
把各轮 log 合并成 `index.parquet` 存 Drive：含 13 项生物指标、QC（FFT 心率/SNR/漏检帧数）、fold。
训练前只看这一个文件就能做划分、算统计，不用读任何 npz。

In [ ]:
# ===== 8. 汇总索引 + 完成度 =====
CLEAN_ERRORS = False        # True → 把 index.parquet 里所有 error 行删掉（历史陈账）

logs = sorted(Path(WORK_DIR, "out").glob("log_*.parquet"))
idx = pd.concat([pd.read_parquet(f) for f in logs], ignore_index=True).drop_duplicates("key", keep="last") if logs else pd.DataFrame()
if len(idx):
    old = os.path.join(DRIVE_ROOT, "index.parquet")
    if os.path.exists(old):
        idx = pd.concat([pd.read_parquet(old), idx], ignore_index=True).drop_duplicates("key", keep="last")
    idx = idx.reset_index(drop=True)
    if "origin" not in idx.columns:
        idx["origin"] = "history"
    idx["origin"] = idx["origin"].fillna("history")
    idx.loc[idx.key.isin(globals().get("_seen", set())), "origin"] = "this-run"
    idx.to_parquet(os.path.join(DRIVE_ROOT, "index.parquet"))
    print("状态:", idx.status.value_counts().to_dict(), "| 累计", len(idx))
    print("来源:", idx.origin.value_counts().to_dict())

    ok = idx[idx.status == "ok"]
    cols = [c for c in ["key", "T", "fps", "hr_ppg", "pulse", "snr", "miss", "fold", "origin"] if c in ok.columns]
    if len(ok) and cols:
        print("\n成功记录：")
        print(ok[cols].head(8).to_string(index=False))

    bad = idx[idx.status != "ok"]
    if len(bad):
        ecols = [c for c in ["key", "err", "ts", "origin"] if c in bad.columns]
        print(f"\n失败记录 {len(bad)} 条（err 决定病因，ts/origin 决定是不是这次跑的）：")
        print(bad[ecols].head(10).to_string(index=False))
        if "origin" in bad.columns:
            new_bad = bad[bad.origin == "this-run"]
            if len(new_bad) == 0:
                print("→ 全是历史陈账（origin=history），这次没有新失败。跑全量时会自动重试它们。")
            else:
                print(f"→ 本次会话新增失败 {len(new_bad)} 条，看 err 列：")
                print(new_bad.err.fillna("").str[:80].value_counts().head(5).to_string())
        if "err" in bad.columns and bad.err.astype(str).str.contains("CUDA", na=False).any():
            print("→ 病因含 CUDA 不可用：切 GPU 或把 LM_DEVICE 设 'cpu'。")
        if CLEAN_ERRORS:
            idx[idx.status == "ok"].to_parquet(old)
            print("\n已清理 error 行，index.parquet 只剩成功记录。")
else:
    print("还没有 log")

for sub in ["roi", "preprocessed", "faces"]:
    d = os.path.join(DRIVE_ROOT, sub)
    n = len(os.listdir(d)) if os.path.isdir(d) else 0
    sz = du(d)
    print(f"{sub:13s} {n:5d} files  {sz:7.3f} GB")

## 8.5 ★★ 全量跑：只改这一格（PRESET + LIMIT）

**先看这个再决定怎么跑**（这台机器 CPU-only，全量的代价必须算清楚）：

| 配置 | 每段帧数 | 单段耗时(CPU 实测外推) | 正脸 1200 段 | 三机位 3600 段 |
|---|---|---|---|---|
| `MAX_FRAMES=300` | 300 | ~1~3 min | 40~60 h | 120~180 h |
| `MAX_FRAMES=600`（官方口径 20 s） | 600 | ~2~5 min | **40~100 h** | 120~300 h |
| `MAX_FRAMES=None`（全时长 3 min） | 2400~4800 | ~10~25 min | 200~500 h | 600~1500 h |

→ **CPU 上只可能跑「300~600 帧」这一档，而且要分很多次会话**。真要全时长/全机位，必须在 GPU 上跑（T4 大约快 15~25 倍）。

下面这格做三件事：① 打印"按你实测速度，剩余要多久"；② 设好全量参数；③ 设会话预算（跑够就停，不会因为你走开而白跑）。

In [ ]:
# ===== 8.5 全量预设（按你的情况选一行，别的别动）=====
#  A) CPU，只正脸，官方口径 20s      → 最现实的 CPU 方案
#  B) CPU，只正脸，全时长            → 除非你打算跑几天
#  C) GPU，三机位，全时长            → 目标形态（换到 Colab/Kaggle 后跑这个）
PRESET = "A"

if PRESET == "A":
    VIEWS, STEPS = ["front"], ["before", "after"]
    MAX_FRAMES, SAVE_OFFICIAL, SAVE_FACES = 600, False, False
elif PRESET == "B":
    VIEWS, MAX_FRAMES = ["front"], None
    SAVE_OFFICIAL, SAVE_FACES = False, False
elif PRESET == "C":
    VIEWS, STEPS = ["front", "left", "right"], ["before", "after"]
    MAX_FRAMES, SAVE_OFFICIAL, SAVE_FACES = None, False, False

LIMIT = 0                    # 0 = 全部
BATCH_PER_ROUND = 20
SESSION_SEC = int(5.5 * 3600)   # 每 5.5 小时自动收工（Colab/长时间会话都安全），跑够就断，重跑继续
SAVE_ROI = True
BATCH = 64 if HAS_CUDA else 8     # GPU 上批大些，FAN 吞吐能翻几倍

# 按"实测均值"重算清单与 ETA（依赖前面已跑出的 index.parquet）
todo_now = db[~db.apply(is_done, axis=1)]
per = None
idx = globals().get("idx")
if idx is not None and "sec" in getattr(idx, "columns", []):
    ok = idx[(idx.status == "ok") & idx.sec.notna()]
    if len(ok):
        per = float(ok.sec.mean())
print(f"预设 {PRESET} | 机位 {VIEWS} | 帧数 {MAX_FRAMES} | 待处理 {len(todo_now)}/{len(db)} 段")
if per:
    print(f"实测 {per:.0f} s/段 → 全量预计 {len(todo_now)*per/3600:.1f} 小时 "
          f"（每会话 {SESSION_SEC/3600:.1f} h，约需 {-(-len(todo_now)*per//SESSION_SEC)} 个会话）")
else:
    print("还没有成功记录的耗时数据，先跑 3~5 段再回来看 ETA")
print("\n下一步顺序：① 回到『3. 工作清单』格重跑（db 按新 VIEWS/STEPS 重建） ② 再回来这格重跑一次看 ETA ③ 然后跑主循环")

# 顺便清掉历史 error 行（可选，让 index.parquet 干净；不清理也会被新结果覆盖）
_p = os.path.join(DRIVE_ROOT, "index.parquet")
if os.path.exists(_p):
    _d = pd.read_parquet(_p)
    print("index 现状:", _d.status.value_counts().to_dict())
    _ok = _d[_d.status == "ok"]
    if len(_ok):
        _ok.to_parquet(_p)
        print("已把 index.parquet 收敛为仅成功行:", len(_ok))

In [ ]:
# ===== 8.6 开跑前确认：把最终生效的参数全部打印出来 =====
# 规则：只有这里显示 LIMIT=0 才是"全量"。不是 0 就别往下跑。
print("=" * 58)
print(f"  LIMIT            = {LIMIT}        {'← 全量 ✅' if LIMIT == 0 else '← 只跑前 N 段 ⚠️'}")
print(f"  VIEWS            = {VIEWS}")
print(f"  STEPS            = {STEPS}")
print(f"  MAX_FRAMES       = {MAX_FRAMES}" + ("   (全时长)" if MAX_FRAMES is None else "   (每段只取前 N 帧)"))
print(f"  SAVE_ROI         = {SAVE_ROI}")
print(f"  SAVE_OFFICIAL    = {SAVE_OFFICIAL}")
print(f"  SAVE_FACES       = {SAVE_FACES}")
print(f"  BATCH_PER_ROUND  = {BATCH_PER_ROUND}")
print(f"  SESSION_SEC      = {SESSION_SEC}  ({SESSION_SEC/3600:.1f} h 自动收工)")
print(f"  LM_DEVICE        = {globals().get('LM_DEVICE')}    (GPU 可用: {globals().get('HAS_CUDA')})")
print(f"  DRIVE_ROOT       = {DRIVE_ROOT}")
print("=" * 58)
n_all = len(db)
n_done = int(db.apply(is_done, axis=1).sum())
print(f"清单 {n_all} 段 | 已完成 {n_done} | 待跑 {n_all - n_done}")
if LIMIT and LIMIT < n_all - n_done:
    print(f"⚠️  注意：LIMIT={LIMIT}，本次只会跑 {LIMIT} 段，不是全量！改第 8.5 格的 LIMIT=0")
print("→ 确认无误后，去跑第『6. 主循环』格")

## 9. 上传到 Hugging Face

**先说清楚两种格子的区别**（你刚才就是混了）：

| 格子类型 | 第一行 | 里面能写什么 |
|---|---|---|
| Python 格（默认） | 空 | Python；`pip` 要写成 `%pip install ...` 或 `!pip install ...`；shell 命令要加 `!` |
| Shell 格 | `%%bash` | 纯 bash；**不要**加 `!`，也**不要**写 python；`pip` 用 `python3 -m pip`（这环境没有 `pip`） |

下面第 9.1 格是**推荐的纯 Python 版**（不碰 shell、不碰环境变量，Colab / 自建 Jupyter 都能跑）。

In [ ]:
# ===== 9.1 纯 Python 上传（推荐）。只需填 DST_REPO，token 可以走 3 种来源 =====
import os
from huggingface_hub import HfApi, login

DST_REPO = ""        # ← 改成 "你的用户名/mcd-rppg-ds"（不存在会自动创建）
PRIVATE  = True
HF_TOKEN = ""        # ← 方式①：直接粘 hf_xxxx（不要有空格/换行）；留空则走下面两种

if not DST_REPO:
    print("⚠️ 先把 DST_REPO 填成 \"你的用户名/mcd-rppg-ds\" 再跑这一格")
else:
    tok = HF_TOKEN.strip() or os.environ.get("HF_TOKEN", "").strip()
    if not tok:
        try:                                  # 方式②：Colab Secrets 里叫 HF_TOKEN
            from google.colab import userdata
            tok = userdata.get("HF_TOKEN")
        except Exception:
            pass
    if tok:                                   # 方式③：都没有就跑 login() 交互输入
        login(token=tok)
    else:
        login()

    api = HfApi()
    api.create_repo(DST_REPO, repo_type="dataset", private=PRIVATE, exist_ok=True)
    print("repo ready →", f"https://huggingface.co/datasets/{DST_REPO}")
    print("开始上传", DRIVE_ROOT, "（文件多，慢是正常的，可以中断后重跑，已存在的会跳过）")
    api.upload_folder(folder_path=DRIVE_ROOT, repo_id=DST_REPO, repo_type="dataset",
                      commit_message="mcd-rppg preprocessed (roi / official npz / faces)")
    print("done →", f"https://huggingface.co/datasets/{DST_REPO}")


In [ ]:
%%bash
# ===== 9.2 Shell 版（可选，等价于 9.1）。注意：%%bash 格里不要写 ! 不要写 python =====
export HF_TOKEN="hf_这里换成你的token"          # 不能有空格/换行
REPO="你的用户名/mcd-rppg-ds"
SRC="/content/drive/MyDrive/mcd_rppg_ds"
python3 -m pip install -q -U "huggingface_hub[cli]" hf_transfer   # 这环境没有 pip 命令，用 python3 -m pip
hf upload "$REPO" "$SRC" . --repo-type=dataset --commit-message "mcd-rppg preprocessed"

In [ ]:
# ===== 9.3 安装 / 排错（Python 格）=====
import shutil, sys, subprocess
print("python:", sys.executable)
print("有 pip 命令吗:", shutil.which("pip") or "没有 → 用 python3 -m pip 或 %pip")
print("有 hf 命令吗:", shutil.which("hf") or "没有 → 用 HfApi（9.1 就不需要它）")

# 需要装包时（取消注释执行）：
# %pip install -q -U "huggingface_hub[cli]" hf_transfer
# 或
# subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub[cli]", "hf_transfer"])

# 检查 token 是否可用（不会打印 token 本身）
import os
tok = os.environ.get("HF_TOKEN", "")
print("环境里的 HF_TOKEN:", ("有，长度 %d" % len(tok)) if tok else "没有")
if tok:
    from huggingface_hub import whoami
    print("我是:", whoami(token=tok)["name"])

---
## 常见坑（都是实测过的）

1. **`ppg_sync` 已经是逐帧对齐的**：`1020_FullHDwebcam_after.avi` 解码 5383 帧，`ppg_sync/1020_FullHDwebcam_after.txt` 5383 行。**不用自己做视频↔PPG 同步。**
2. **`ppg_sync_npy` 极性是反的**（`col0 + npy ≡ 236`）。官方 csv 的 ppg 统计对应 **txt 第 0 列** → 只读 txt。
3. **三机位帧率不同**：FullHDwebcam 29.97 / USBVideo 30 / IriunWebcam **24**。每段 npz 都存了 `fps`，训练时窗口按"秒"折算。
4. **官方 npz 只用了前 600 帧（20 s）**，3 分钟里扔掉 140 秒。`MAX_FRAMES=None` 就是把这部分补回来（数据量 ×8）。
5. 视频有坏帧（ffprobe 报 `Error at MB`），解码帧数与 ppg 行数偶尔差几帧 → 代码用 `T = len(帧)` 并在 `info["err"]` 里记下来。
6. **Drive 写小文件偏慢**（每段 1~3 个文件，约 0.3~2 s），已经用"本地写完再 copy"的方式减少随机写。
7. Colab 免费版 90 分钟无操作会断 → 每轮 N 段就落盘，重跑这一格自动续。
8. 内存：整段 640×480×3 的 5000 帧是 4.6 GB，**不要**一次性读进内存，本 notebook 全程是流式 batch，峰值 < 1 GB。
9. **`AssertionError: Torch not compiled with CUDA enabled`**：先看 `torch.__version__` 有没有 **`+cpu`** 后缀（有 = CPU-only wheel），再看 `nvidia-smi` 是否存在（不存在 = 这台机器根本没有 NVIDIA 驱动）。三个可能——
   ① 运行时是 CPU 型（`运行时 → 更改运行时类型 → T4 GPU`）；
   ② `pip install` 时把 torch 换成了 CPU wheel（`import torch; torch.version.cuda` 会是 `None`）→
   重装：`pip install -q --force-reinstall torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121`；
   ③ 装了 torch 但驱动不匹配。
   本 notebook 现在会自动探测：没有 GPU 就退回 `LM_DEVICE="cpu"` 并自动把 `MAX_FRAMES` 收到 600、`BATCH` 收到 8（慢但能跑），
   同时在汇总格直接告诉你是 CUDA 问题。
10. **"只跑了一轮"**：轮数 = `ceil(待处理段数 / BATCH_PER_ROUND)`。`LIMIT=5` 就是 1 轮。
